# Medical Imaging ML Project: Scope Declaration
* **Anatomical Region:** Bone (Hand and Wrist)
* **Imaging Modality:** X-ray (Radiographs)
* **Problem Type:** Classification (Age Group Grouping)
* **Target Classes:** Phase 1 (Early Growth), Phase 2 (Mid-Stage), Phase 3 (Mature Bone)
* **Dataset Source & Link:** RSNA Bone Age Dataset (Kaggle Mirror)


**Goal:** Checking if an AI system can accurately categorize a child's bone development stage from a hand X-ray?

**Intended User:** Pediatricians and medical students checking for child growth disorders.


## Step 1: Import Libraries
We import **TensorFlow** to build our neural network, **OpenCV** to process our images, and standard data tools like **NumPy** and **Matplotlib**.


In [1]:
%pip install tensorflow

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

print("Libraries imported successfully!")
print("TensorFlow Version:", tf.__version__)


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


I0000 00:00:1789447478.179617   81197 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1789447478.218658   81197 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789447479.744214   81197 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Libraries imported successfully!
TensorFlow Version: 2.21.0


## Step 2: Dataset Configuration
We point the notebook directly to the files.


In [2]:

import os

# 1. Get the absolute path to your home directory automatically
home = os.path.expanduser("~")

# 2. Point exactly to where your files are 
CSV_PATH = os.path.join(home, "Downloads", "archive (14)", "boneage-training-dataset","boneage-training-dataset/")
IMAGES_DIR = os.path.join(home, "Downloads", "archive (14)", "boneage-training-dataset", "boneage-training-dataset/")

# 3. Check if the files can be found"boneage-training-dataset
if os.path.exists(CSV_PATH) and os.path.exists(IMAGES_DIR):
    print("Success! Both your CSV file and Image folder were found.")
else:
    print("Warning: Still cannot find the files.")
    print("Looking for CSV at:", CSV_PATH)
    print("Looking for Images at:", IMAGES_DIR)


Success! Both your CSV file and Image folder were found.


## Step 3: Preprocessing Pipeline
* We build a standard computer vision function. Whenever we pass an image path to this function, it opens the image in black-and-white, resizes it to `128 \times 128` pixels so the model works quickly, and divides by `255.0` to normalize the pixels between `0.0` and `1.0`.


In [3]:
def process_single_image(image_path):
    # 1. Read image as Grayscale (black and white)
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    
    # 2. Resize to 128x128 pixels
    img_resized = cv2.resize(img, (128, 128))
    
    # 3. Normalize values between 0.0 and 1.0
    img_normalized = img_resized.astype(np.float32) / 255.0
    
    # 4. Add channel dimension for the model -> (128, 128, 1)
    img_final = np.expand_dims(img_normalized, axis=-1)
    
    return img_final

print("Image processing function created successfully.")


Image processing function created successfully.


## Step 4: Load Dataset Using the CSV Metadata 
* We open the CSV file with Pandas. We create 3 development classes based on the child's bone age in months (Phase 0, 1, or 2). Then, we loop through the images, run them through our Step 3 pipeline, and save them into training matrices (`X_data` and `y_data`).


In [4]:

import os
import pandas as pd
import numpy as np

# 1. Direct explicit path definitions to completely bypass cell memory issues
home = os.path.expanduser("~")
base_folder = os.path.join(home, "Downloads", "archive (14)")

#
CELL_CSV_PATH = os.path.join(base_folder, "boneage-training-dataset.csv")
CELL_IMAGES_DIR = os.path.join(base_folder, "boneage-training-dataset", "boneage-training-dataset")

# 2. Read the CSV text log table file
df = pd.read_csv(CELL_CSV_PATH)

# 3. Categorize clinical ages into 3 clear growth phases
def assign_growth_class(months):
    if months < 100:
        return 0  # Phase 0: Early Growth Stage
    elif 100 <= months <= 160:
        return 1  # Phase 1: Mid-Stage Development
    else:
        return 2  # Phase 2: Mature Bone Structure

df['label'] = df['boneage'].apply(assign_growth_class)

processed_images_list = []
final_labels_list = []

# Process the first 200 items to keep your prototype fast and smooth
max_samples = min(200, len(df))

for idx, row in df.head(max_samples).iterrows():
    image_id = str(row['id'])
    filename = f"{image_id}.png" if not image_id.endswith('.png') else image_id
    full_image_path = os.path.join(CELL_IMAGES_DIR, filename)
    
    # Only load if the actual image file exists in the folder
    if os.path.exists(full_image_path):
        # Calls your Step 3 computer vision preprocessing logic
        processed_tensor = process_single_image(full_image_path)
        processed_images_list.append(processed_tensor)
        final_labels_list.append(row['label'])

# 4. Save into clean numpy matrices for model training
X_data = np.array(processed_images_list)
y_data = np.array(final_labels_list)

print("--- Real Matched Dataset Loading Complete ---")
print("Image Matrix Shape (X_data):", X_data.shape)
print("Label Matrix Shape (y_data):", y_data.shape)
print("SUCCESS: Data matrices are loaded and ready for Step 5!")



--- Real Matched Dataset Loading Complete ---
Image Matrix Shape (X_data): (200, 128, 128, 1)
Label Matrix Shape (y_data): (200,)
SUCCESS: Data matrices are loaded and ready for Step 5!


## Step 5: Data Splitting Strategy
* We divide our data so that **80%** goes into training the model. The remaining 20% is split perfectly in half: **10%** for validation tracking during training, and **10%** for the final unseen test evaluations.


In [5]:
# First split: Separate exactly 80% for Training, leaving 20% temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    X_data, y_data, test_size=0.20, random_state=42, stratify=y_data
)

# Second split: Divide that remaining 20% completely in half (10% Val, 10% Test)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Training matrices (80%): {X_train.shape}")
print(f"Validation matrices (10%): {X_val.shape}")
print(f"Testing matrices (10%): {X_test.shape}")


Training matrices (80%): (160, 128, 128, 1)
Validation matrices (10%): (20, 128, 128, 1)
Testing matrices (10%): (20, 128, 128, 1)


## Step 6: Model Architecture & Design
We are building a simple Convolutional Neural Network (CNN). It looks at small sections of the X-ray to pick up bone edges, structures, and shapes.


In [6]:
model = models.Sequential([
    # Input Layer matching our 128x128 grayscale images
    layers.Input(shape=(128, 128, 1)),
    
    # Block 1: Edges
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Block 2: Structural Shapes
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Flattening data into a vector and adding Dropout to prevent overfitting
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    
    # Output Layer: 3 nodes for our 3 bone development categories
    layers.Dense(3, activation='softmax')
])

print("Computer Vision AI Model built successfully!")
model.summary()


Computer Vision AI Model built successfully!


E0000 00:00:1789447483.693857   81197 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 57600)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     3,686,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,705,475 (14.14 MB)

 Trainable params: 3,705,475 (14.14 MB)

 Non-trainable params: 0 (0.00 B)

## Step 7: Training Configuration & Execution 
We assign standard training helpers (`Adam` optimizer and `crossentropy` loss calculations) and train our model for $5$ loops (epochs).


In [7]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Starting model training operations...")
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=8,
    validation_data=(X_val, y_val)
)


Starting model training operations...
Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.3187 - loss: 1.2548 - val_accuracy: 0.6500 - val_loss: 1.0105
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.4062 - loss: 1.0409 - val_accuracy: 0.5500 - val_loss: 1.0141
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.5375 - loss: 0.9252 - val_accuracy: 0.5500 - val_loss: 0.9462
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - accuracy: 0.5312 - loss: 0.8860 - val_accuracy: 0.5500 - val_loss: 0.9326
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.6438 - loss: 0.7685 - val_accuracy: 0.6000 - val_loss: 0.9104


## Step 8: Final Evaluation & Metrics Report 
We test the model on our untouched **Test Set** to see how accurately it categorizes bone maturity.


In [8]:
# Calculate loss and accuracy numbers
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy Score: {accuracy * 100:.2f}%")

# Generate predictions to check class performance
raw_predictions = model.predict(X_test, verbose=0)
predicted_classes = np.argmax(raw_predictions, axis=1)

# Display a neat performance report
print("\n--- Model Performance Report per Category ---")
print(classification_report(y_test, predicted_classes, target_names=['Early', 'Mid-Stage', 'Mature']))


Test Accuracy Score: 65.00%

--- Model Performance Report per Category ---
              precision    recall  f1-score   support

       Early       0.67      0.89      0.76         9
   Mid-Stage       0.62      0.56      0.59         9
      Mature       0.00      0.00      0.00         2

    accuracy                           0.65        20
   macro avg       0.43      0.48      0.45        20
weighted avg       0.58      0.65      0.61        20



/home/student/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/student/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/student/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## Step 9: Model Export Routine 
We save our trained model settings to a single hard drive file called `bone_age_model.h5`. Our web app will read this file later.


In [9]:
# Save the model architecture and weights
model.save("bone_age_model.h5")

print("Success! 'bone_age_model.h5' file saved and ready for Streamlit deployment use.")


Success! 'bone_age_model.h5' file saved and ready for Streamlit deployment use.


## Step 10: Streamlit Diagnostic Portal Setup 
Create a new file on our computer named `app.py`.


In [10]:
pip install streamlit


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:

import inspect
import subprocess

def build_diagnostic_portal():
    import streamlit as st
    import tensorflow as tf
    import numpy as np
    import cv2
    from PIL import Image

    st.set_page_config(page_title="Bone Maturity Portal", layout="centered")

    st.title("Pediatric Bone Maturity Diagnostic Portal")
    st.write("Upload a child's hand X-ray to predict their skeletal growth category phase.")
    st.warning(" Coursework Prototype Notice: This system is an educational coursework prototype and is NOT a medical device.")

    @st.cache_resource
    def load_my_model():
        return tf.keras.models.load_model("bone_age_model.h5")

    try:
        model = load_my_model()
        st.success("AI Model loaded successfully!")
    except Exception as e:
        st.error("Could not find 'bone_age_model.h5'. Make sure you ran Step 9 first.")

    uploaded_file = st.file_uploader("Upload an X-ray scan...", type=["png", "jpg", "jpeg"])

    if uploaded_file is not None:
        pil_image = Image.open(uploaded_file).convert("L")
        st.image(pil_image, caption="Uploaded Scan Preview", width=300)
        
        opencv_img = np.array(pil_image)
        resized_img = cv2.resize(opencv_img, (128, 128))
        normalized_img = resized_img.astype(np.float32) / 255.0
        input_tensor = np.expand_dims(normalized_img, axis=(0, -1))
        
        prediction_scores = model.predict(input_tensor)
        predicted_class_index = int(np.argmax(prediction_scores))
        
        
        confidence_value = float(prediction_scores[0][predicted_class_index] * 100)
        
        categories = ["Phase 0: Early Growth Phase", "Phase 1: Mid-Stage Development", "Phase 2: Mature Bone Structure"]
        
        st.subheader("Diagnostic Assessment Result:")
        st.write(f"**Classification:** {categories[predicted_class_index]}")
        st.write(f"**Confidence Index:** {confidence_value:.2f}%")
        st.info(f"**Plain Language Summary:** The model is {confidence_value:.1f}% sure that this hand scan shows features matching {categories[predicted_class_index]}.")

# Extract the lines directly from the code-colored function structure above
raw_lines = inspect.getsource(build_diagnostic_portal).splitlines()[1:]
cleaned_lines = [line[4:] if line.startswith("    ") else line for line in raw_lines]

with open("app.py", "w") as f:
    f.write("\n".join(cleaned_lines))

# Launch with config flags to skip email validation forms
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--global.developmentMode=false",
    "--server.headless=true"
])

print("PIPELINE STACK READY: 'app.py' file completely updated!")


PIPELINE STACK READY: 'app.py' file completely updated!
